# Fantastic Breaks — Synthetic→Real Transfer

Đánh giá model DiffusionNet (đã pretrain trên Breaking Bad, IoU 0.44) trên **dữ liệu vỡ THẬT**.
Ba thí nghiệm trên **cùng tập test FB**:
1. **Zero-shot** — model BB chạy thẳng, không train (đo domain gap).
2. **Fine-tune** — model BB → fine-tune trên FB (phương pháp chính).
3. **From-scratch** — train chỉ trên FB (baseline cho thấy 150 mẫu là không đủ).

**Trước khi chạy** — Settings: GPU + Internet ON. Add Input **2 dataset**:
- `Fantastic_Breaks_v1` (mesh thật + nhãn mask)
- dataset chứa checkpoint Breaking Bad `diffnet_best_256.pt` (upload file 1.8MB thành 1 Kaggle Dataset)

## 1. Setup

In [ ]:
import os, sys, random, json, time
from pathlib import Path
from collections import deque, defaultdict

!pip install -q trimesh potpourri3d robust_laplacian plyfile scikit-learn fast-simplification

DIFFNET_DIR = '/kaggle/working/diffusion-net'
if not os.path.exists(DIFFNET_DIR):
    !git clone https://github.com/nmwsharp/diffusion-net.git {DIFFNET_DIR}
sys.path.append(f'{DIFFNET_DIR}/src')

import numpy as np
import torch
import torch.nn.functional as F
import trimesh
import fast_simplification
from scipy.spatial import cKDTree
import diffusion_net

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
if not torch.cuda.is_available():
    print('⚠️ KHÔNG thấy GPU! Settings → Accelerator → GPU.')
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

## 2. Cấu hình — PHẢI khớp kiến trúc Breaking Bad để load được pretrained

In [ ]:
# --- Kiến trúc (khớp model BB diffnet_best_256.pt: thực chất C_width=128) ---
INPUT_FEATURES = 'xyz_hks_curv'
N_HKS   = 16
K_EIG   = 128
C_WIDTH = 128
N_BLOCK = 4
C_IN = (3 if 'xyz' in INPUT_FEATURES else 0) + (N_HKS if 'hks' in INPUT_FEATURES else 0) + (1 if 'curv' in INPUT_FEATURES else 0)

# --- FB preprocessing ---
DECIMATE_VERTS = 22000      # mesh FB ~1.2M vertex -> giảm còn ~22k
DILATE_RINGS   = 1          # nở dải vỡ 1 ring (giữ dải mỏng sau decimate)

# --- Split + train ---
N_TEST         = 50         # số mẫu test (giữ cố định cho cả 3 thí nghiệm)
FT_LR          = 1e-4       # fine-tune: lr nhỏ
FT_EPOCHS      = 25         # nâng 12 -> 25, early-stop theo best val IoU
SCRATCH_LR     = 1e-3
SCRATCH_EPOCHS = 25
AUGMENT        = True       # xoay ngẫu nhiên xyz khi train -> giảm overfit trên 100 mesh
CE_CLASS_WEIGHTS = [1.0, 3.0]
DICE_WEIGHT    = 1.0
LOSS_MODE      = 'ce_dice'
GRAD_CLIP      = 1.0

OP_CACHE = '/kaggle/temp/fb_op_cache'    # operator cache (ephemeral)
FB_CACHE = '/kaggle/temp/fb_mesh_cache'  # mesh đã decimate + nhãn
print(f'C_in={C_IN}  decimate->{DECIMATE_VERTS}  test={N_TEST}  augment={AUGMENT}  ft_epochs={FT_EPOCHS}')

## 3. Dò dataset FB + checkpoint Breaking Bad

In [ ]:
INPUT = Path('/kaggle/input')
print('Datasets attached:')
for d in INPUT.iterdir():
    print('  -', d.name)

# FB: mỗi mẫu là 1 folder chứa meta_*.npz + model_b_*.ply
meta_files = list(INPUT.rglob('meta_*.npz'))
sample_dirs = sorted(set(m.parent for m in meta_files))
assert len(sample_dirs) > 0, '❌ Không thấy meta_*.npz — kiểm tra dataset Fantastic_Breaks_v1 đã attach.'
print(f'\n✅ Fantastic Breaks: {len(sample_dirs)} mẫu')

# Checkpoint BB: file .pt
pt_files = [p for p in INPUT.rglob('*.pt')]
assert len(pt_files) > 0, '❌ Không thấy file .pt — upload diffnet_best_256.pt thành 1 Kaggle Dataset rồi Add Input.'
BB_CKPT = str(pt_files[0])
print(f'✅ BB checkpoint: {BB_CKPT}')

## 4. Tiền xử lý FB: decimate + chuyển nhãn `mask` + nở dải + normalize

Nhãn lấy TRỰC TIẾP từ `meta.mask` (nhãn thủ công), chỉ cần chuyển sang mesh đã giảm.

In [ ]:
def preprocess_fb(sample_dir, target_verts, dilate_rings):
    """1 mẫu FB -> (verts normalized, faces, labels) hoặc None."""
    sample_dir = Path(sample_dir)
    mb = sorted(sample_dir.glob('model_b_*.ply'))
    mt = sorted(sample_dir.glob('meta_*.npz'))
    if not mb or not mt:
        return None
    m = trimesh.load(str(mb[0]), process=False)
    V0 = np.ascontiguousarray(np.asarray(m.vertices, dtype=np.float64))
    F0 = np.ascontiguousarray(np.asarray(m.faces, dtype=np.int32))
    mask = np.load(str(mt[0]))['mask'].astype(bool)
    if len(V0) != len(mask) or len(F0) < 10:
        return None
    # 1) Quadric decimation -> ~target_verts (target theo số tam giác ~ 2x verts)
    target_tris = int(target_verts * 2)
    reduction = max(0.05, min(0.99, 1.0 - target_tris / len(F0)))
    Vd, Fd = fast_simplification.simplify(V0, F0, target_reduction=reduction)
    Vd = np.asarray(Vd, dtype=np.float32); Fd = np.asarray(Fd, dtype=np.int64)
    if len(Vd) < 100 or len(Fd) < 100:
        return None
    # 2) Chuyển nhãn: mỗi vertex mới -> nhãn của vertex gốc gần nhất
    _, idx = cKDTree(V0).query(Vd, k=1)
    labels = mask[idx].astype(np.int64)
    # 3) Nở dải vỡ N ring
    if dilate_rings > 0:
        adj = defaultdict(set)
        for a, b, c in Fd:
            adj[a].update((b, c)); adj[b].update((a, c)); adj[c].update((a, b))
        for _ in range(dilate_rings):
            grow = set()
            for v in np.where(labels == 1)[0]:
                grow.update(adj[int(v)])
            if grow:
                labels[list(grow)] = 1
    # 4) Normalize (khớp pipeline BB)
    Vd = Vd - Vd.mean(0)
    sc = np.linalg.norm(Vd.max(0) - Vd.min(0))
    if sc > 0:
        Vd = Vd / sc
    return Vd.astype(np.float32), Fd, labels

# Xử lý toàn bộ + cache
Path(FB_CACHE).mkdir(parents=True, exist_ok=True)
cache_files, fracs = [], []
t0 = time.time()
for i, sd in enumerate(sample_dirs):
    out = Path(FB_CACHE) / f'fb_{i:04d}.npz'
    if out.exists():
        cache_files.append(out)
        fracs.append(float(np.load(out)['labels'].mean())); continue
    res = preprocess_fb(sd, DECIMATE_VERTS, DILATE_RINGS)
    if res is None:
        continue
    v, f, lab = res
    np.savez(out, verts=v, faces=f, labels=lab)
    cache_files.append(out); fracs.append(float(lab.mean()))
    if (i + 1) % 25 == 0:
        print(f'  {i+1}/{len(sample_dirs)}  ({time.time()-t0:.0f}s)')
print(f'\n✅ Preprocessed {len(cache_files)} mẫu trong {time.time()-t0:.0f}s')
print(f'   Fracture trung bình: {100*np.mean(fracs):.1f}%')

## 5. Dataset + features (khớp HỆT Breaking Bad)

In [ ]:
from torch.utils.data import Dataset

class FBDataset(Dataset):
    def __init__(self, files, op_cache_dir, k_eig):
        self.files = list(files)
        self.op = Path(op_cache_dir); self.op.mkdir(parents=True, exist_ok=True)
        self.k = k_eig
    def __len__(self):
        return len(self.files)
    def __getitem__(self, i):
        d = np.load(self.files[i])
        verts = torch.tensor(d['verts'], dtype=torch.float32)
        faces = torch.tensor(d['faces'], dtype=torch.long)
        labels = torch.tensor(d['labels'], dtype=torch.long)
        try:
            frames, mass, L, evals, evecs, gradX, gradY = diffusion_net.geometry.get_operators(
                verts, faces, k_eig=self.k, op_cache_dir=str(self.op))
        except Exception as e:
            print('⚠️ op fail', self.files[i].name, e); return None
        return {'verts': verts, 'faces': faces, 'labels': labels,
                'mass': mass, 'L': L, 'evals': evals, 'evecs': evecs,
                'gradX': gradX, 'gradY': gradY}

def compute_metrics(preds, labels):
    preds = preds.flatten(); labels = labels.flatten()
    tp = ((preds==1)&(labels==1)).sum().item(); fp = ((preds==1)&(labels==0)).sum().item()
    fn = ((preds==0)&(labels==1)).sum().item(); tn = ((preds==0)&(labels==0)).sum().item()
    tot = tp+fp+fn+tn
    acc = (tp+tn)/tot if tot else 0
    prec = tp/(tp+fp) if (tp+fp) else 0
    rec = tp/(tp+fn) if (tp+fn) else 0
    f1 = 2*prec*rec/(prec+rec) if (prec+rec) else 0
    iou = tp/(tp+fp+fn) if (tp+fp+fn) else 0
    return {'acc': acc, 'precision': prec, 'recall': rec, 'f1': f1, 'iou': iou}

def _std_log(x):
    x = torch.log(x.clamp_min(1e-8))
    return (x - x.mean(0, keepdim=True)) / (x.std(0, keepdim=True) + 1e-6)

def get_features(sample):
    verts = sample['verts'].to(device)
    feats = []
    if 'xyz' in INPUT_FEATURES:
        feats.append(verts)
    if 'hks' in INPUT_FEATURES:
        hks = diffusion_net.geometry.compute_hks_autoscale(
            sample['evals'].to(device), sample['evecs'].to(device), N_HKS)
        feats.append(_std_log(hks))
    if 'curv' in INPUT_FEATURES:
        L = sample['L'].to(device); mass = sample['mass'].to(device)
        Hvec = torch.sparse.mm(L, verts) / mass.clamp_min(1e-8).unsqueeze(-1)
        feats.append(_std_log(Hvec.norm(dim=-1, keepdim=True)))
    return feats[0] if len(feats) == 1 else torch.cat(feats, dim=-1)

def forward_sample(model, sample):
    return model(x_in=get_features(sample),
        mass=sample['mass'].to(device), L=sample['L'].to(device),
        evals=sample['evals'].to(device), evecs=sample['evecs'].to(device),
        gradX=sample['gradX'].to(device), gradY=sample['gradY'].to(device),
        faces=sample['faces'].to(device))

# ---- Augmentation: xoay ngẫu nhiên (chỉ tác động kênh xyz; HKS/curvature bất biến) ----
def _random_rotation():
    A = torch.randn(3, 3)
    Q, _ = torch.linalg.qr(A)
    if torch.det(Q) < 0:
        Q[:, 0] = -Q[:, 0]          # đảm bảo ma trận xoay đúng (det=+1)
    return Q.float()

def augment_sample(sample):
    R = _random_rotation()
    s = dict(sample)                # copy nông; chỉ thay verts
    s['verts'] = sample['verts'] @ R.t()
    return s

ce_weights = torch.tensor(CE_CLASS_WEIGHTS, device=device)
ce_fn = torch.nn.CrossEntropyLoss(weight=ce_weights)
def dice_loss(logits, labels, eps=1e-6):
    probs = F.softmax(logits, dim=-1)[:, 1]; target = (labels == 1).float()
    inter = (probs*target).sum()
    return 1.0 - (2.0*inter + eps) / (probs.sum() + target.sum() + eps)
def compute_loss(logits, labels):
    return ce_fn(logits, labels) + DICE_WEIGHT * dice_loss(logits, labels)

def train_epoch(model, dataset, optimizer, augment=False):
    model.train(); idxs = list(range(len(dataset))); random.shuffle(idxs)
    losses = []
    for j in idxs:
        s = dataset[j]
        if s is None: continue
        if augment:
            s = augment_sample(s)
        optimizer.zero_grad()
        loss = compute_loss(forward_sample(model, s), s['labels'].to(device))
        loss.backward()
        if GRAD_CLIP is not None:
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step(); losses.append(loss.item())
    return float(np.mean(losses)) if losses else 0.0

@torch.no_grad()
def evaluate(model, dataset):
    model.eval(); ps, ls = [], []
    for j in range(len(dataset)):
        s = dataset[j]
        if s is None: continue
        ps.append(forward_sample(model, s).argmax(-1).cpu()); ls.append(s['labels'])
    return compute_metrics(torch.cat(ps), torch.cat(ls))

## 6. Chia train/test + hàm load model BB

In [ ]:
rng = random.Random(SEED)
files = list(cache_files); rng.shuffle(files)
test_files = files[:N_TEST]
train_files = files[N_TEST:]
print(f'Train (fine-tune/from-scratch): {len(train_files)}  |  Test: {len(test_files)}')

train_ds = FBDataset(train_files, OP_CACHE, K_EIG)
test_ds  = FBDataset(test_files,  OP_CACHE, K_EIG)

def build_model():
    return diffusion_net.layers.DiffusionNet(C_in=C_IN, C_out=2, C_width=C_WIDTH,
            N_block=N_BLOCK, outputs_at='vertices', dropout=True).to(device)

def load_bb(model):
    ck = torch.load(BB_CKPT, map_location=device, weights_only=False)
    model.load_state_dict(ck['model_state_dict'])
    print(f'  Loaded BB weights (best_val_iou={ck.get("best_val_iou")})')
    return model

# Build op cache cho test 1 lần (lần đầu chậm)
print('Warming operator cache (test set)...')
_ = evaluate(build_model(), test_ds)
print('OK')

## 7. Ba thí nghiệm

In [ ]:
results = {}

# --- 1) ZERO-SHOT: model BB chạy thẳng, không train ---
print('=== 1) ZERO-SHOT ===')
m_zs = load_bb(build_model())
results['zero_shot'] = evaluate(m_zs, test_ds)
print('  ', {k: round(v, 4) for k, v in results['zero_shot'].items()})

# --- 2) FINE-TUNE từ BB (có augmentation) ---
print('\n=== 2) FINE-TUNE từ BB ===')
m_ft = load_bb(build_model())
opt = torch.optim.Adam(m_ft.parameters(), lr=FT_LR)
best_ft, best_ft_state, no_improve = 0, None, 0
for ep in range(FT_EPOCHS):
    tl = train_epoch(m_ft, train_ds, opt, augment=AUGMENT)
    mv = evaluate(m_ft, test_ds)
    print(f'  ft ep{ep+1}/{FT_EPOCHS} loss={tl:.4f} IoU={mv["iou"]:.3f} F1={mv["f1"]:.3f} P={mv["precision"]:.3f} R={mv["recall"]:.3f}')
    if mv['iou'] > best_ft:
        best_ft = mv['iou']; best_ft_state = {k: v.cpu().clone() for k, v in m_ft.state_dict().items()}; no_improve = 0
    else:
        no_improve += 1
    if no_improve >= 8:    # early stopping
        print('  early stop (no improvement 8 epochs)'); break
m_ft.load_state_dict(best_ft_state)
results['fine_tune'] = evaluate(m_ft, test_ds)

# --- 3) FROM-SCRATCH (chỉ FB, random init, cùng augmentation để so công bằng) ---
print('\n=== 3) FROM-SCRATCH ===')
m_sc = build_model()
opt = torch.optim.Adam(m_sc.parameters(), lr=SCRATCH_LR)
best_sc, best_sc_state, no_improve = 0, None, 0
for ep in range(SCRATCH_EPOCHS):
    tl = train_epoch(m_sc, train_ds, opt, augment=AUGMENT)
    mv = evaluate(m_sc, test_ds)
    print(f'  sc ep{ep+1}/{SCRATCH_EPOCHS} loss={tl:.4f} IoU={mv["iou"]:.3f} F1={mv["f1"]:.3f}')
    if mv['iou'] > best_sc:
        best_sc = mv['iou']; best_sc_state = {k: v.cpu().clone() for k, v in m_sc.state_dict().items()}; no_improve = 0
    else:
        no_improve += 1
    if no_improve >= 8:
        print('  early stop (no improvement 8 epochs)'); break
m_sc.load_state_dict(best_sc_state)
results['from_scratch'] = evaluate(m_sc, test_ds)

## 8. Bảng kết quả + lưu

In [ ]:
print('\n================  KẾT QUẢ TRÊN FANTASTIC BREAKS (test {} mẫu)  ================'.format(len(test_files)))
print(f'{"Setting":<16}{"Acc":>8}{"Prec":>8}{"Recall":>8}{"F1":>8}{"IoU":>8}')
for name in ['zero_shot', 'from_scratch', 'fine_tune']:
    m = results[name]
    print(f'{name:<16}{m["acc"]:>8.3f}{m["precision"]:>8.3f}{m["recall"]:>8.3f}{m["f1"]:>8.3f}{m["iou"]:>8.3f}')

import json, os
os.makedirs('/kaggle/working/fb_results', exist_ok=True)
torch.save({'model_state_dict': best_ft_state, 'iou': results['fine_tune']['iou']},
           '/kaggle/working/fb_results/diffnet_fb_finetuned.pt')
with open('/kaggle/working/fb_results/fb_metrics.json', 'w') as fp:
    json.dump(results, fp, indent=2)
print('\n✅ Saved /kaggle/working/fb_results/  (model fine-tuned + metrics)')

from IPython.display import FileLink
FileLink('fb_results/fb_metrics.json')

## 9. Mô phỏng kết quả: dự đoán + tách mảnh (explode) để lộ bề mặt vỡ

Tô **đỏ = vùng model dự đoán là fracture**, xám = original. Nếu mesh có **nhiều mảnh** (vd Breaking Bad) thì tự **đẩy các mảnh ra xa** để nhìn thấy mặt cắt bên trong; mẫu Fantastic Breaks là 1 mảnh nên mặt vỡ đã lộ sẵn ở rìa. Đổi `show='gt'` để xem nhãn thật, hoặc tăng `explode` để tách rộng hơn.

In [ ]:
import plotly.graph_objects as go

def _components(faces, n):
    """Union-find: id thành phần liên thông của mỗi vertex."""
    parent = np.arange(n)
    def find(x):
        root = x
        while parent[root] != root: root = parent[root]
        while parent[x] != root:
            parent[x], x = root, parent[x]
        return root
    for a, b, c in faces:
        for u, w in ((int(a), int(b)), (int(b), int(c))):
            ru, rw = find(u), find(w)
            if ru != rw: parent[ru] = rw
    return np.array([find(i) for i in range(n)])

@torch.no_grad()
def simulate(sample, model, explode=0.3, show='pred'):
    """Dự đoán fracture + tách mảnh (nếu >1) để lộ bề mặt vỡ. show='pred'|'gt'."""
    model.eval()
    pred = forward_sample(model, sample).argmax(-1).cpu().numpy()
    gt   = sample['labels'].numpy()
    v = sample['verts'].numpy().copy(); f = sample['faces'].numpy()
    comp = _components(f, len(v)); ids = np.unique(comp); gc = v.mean(0)
    if len(ids) > 1:                       # nhiều mảnh -> đẩy ra xa tâm (explode)
        for cid in ids:
            mk = comp == cid
            d = v[mk].mean(0) - gc; nrm = np.linalg.norm(d)
            if nrm > 1e-6:
                v[mk] += (d / nrm) * explode
    color = (pred if show == 'pred' else gt).astype(np.float32)
    m = compute_metrics(torch.tensor(pred), torch.tensor(gt))
    fig = go.Figure(go.Mesh3d(
        x=v[:,0], y=v[:,1], z=v[:,2], i=f[:,0], j=f[:,1], k=f[:,2],
        intensity=color, colorscale=[[0,'lightgray'],[1,'crimson']],
        cmin=0, cmax=1, flatshading=True, showscale=False))
    fig.update_layout(width=820, height=620, scene=dict(aspectmode='data'),
        title=f'{"Prediction" if show=="pred" else "Ground Truth"} (đỏ=fracture) — '
              f'{len(ids)} mảnh, IoU={m["iou"]:.3f} F1={m["f1"]:.3f}')
    fig.show()

# Mô phỏng vài mẫu TEST bằng model fine-tuned (m_ft). Đổi show='gt' để đối chiếu.
# Lưu ý: mẫu FB là 1 mảnh -> mặt vỡ đã lộ sẵn, chỉ tô đỏ. (explode chỉ tách khi >1 mảnh)
for idx in [0, 1, 2, 3]:
    s = test_ds[idx]
    if s is None: continue
    print(f'--- test sample {idx} ---')
    simulate(s, m_ft, explode=0.3, show='pred')